# 08c · The store: one mmap'd file, one binary search, in C (paper §4)
**Requires the C kernel.** Extracted verbatim at build time: `monad3_lookup` (`ptol.c` 136–168), `gr_log_anchor` (193–202), `monad3_gamma_radial` (206–213), and `monad_word_coords` (`monad.c` 170–220). The header layout is `PtolC/monad3c.h`, included directly.

In [1]:
//%cflags:-I../../PtolC
#include <stdio.h>
#include <stdint.h>
#include <fcntl.h>
#include <sys/mman.h>
#include <sys/stat.h>
#include <unistd.h>
#include <string.h>
#include "monad3c.h"

int main(void) {
    int fd = open("../../PtolC/monad3_c.bin", O_RDONLY); struct stat st; fstat(fd, &st);
    const Monad3cHeader *h = mmap(NULL, st.st_size, PROT_READ, MAP_PRIVATE, fd, 0);
    printf("file size      %lld bytes\n", (long long)st.st_size);
    printf("magic          %.7s   version %u\n", h->magic, h->version);
    printf("words          %u   (one WordRec per word: 16 bytes)\n", h->n_words);
    printf("english store  %u entries   wordnet store  %u entries   phonetic store  %u entries\n", h->n_eng, h->n_wn, h->n_phon);
    printf("co-occurrence  %u edges\n", h->nnz);
    printf("%s: header magic is MONAD3C and sizeof(WordRec) == 16\n", (memcmp(h->magic, MONAD3C_MAGIC, 8) == 0 && sizeof(WordRec) == 16) ? "PASS" : "FAIL");
    return 0;
}


file size      48209731 bytes
magic          MONAD3C   version 1
words          347119   (one WordRec per word: 16 bytes)
english store  164283 entries   wordnet store  146743 entries   phonetic store  123455 entries
co-occurrence  2248064 edges
PASS: header magic is MONAD3C and sizeof(WordRec) == 16


## One binary search gives the indices into all three stores

In [2]:
//%cflags:-I../../PtolC
//%ldflags:-lm
#include <stdio.h>
#include <stdint.h>
#include <fcntl.h>
#include <sys/mman.h>
#include <sys/stat.h>
#include <unistd.h>
#include <string.h>
#include <ctype.h>
#include <math.h>
#include "monad3c.h"

static const unsigned char *g_m3 = NULL; static const Monad3cHeader *g_m3h = NULL;

/* ptol.c:136-168 */
static int monad3_lookup(const char *word, int32_t out[3], uint32_t *degree)
{
    out[0] = out[1] = out[2] = -1;
    if (degree) *degree = 0;
    if (!g_m3h) return 0;

    char w[128];
    size_t j = 0;
    for (const char *p = word; *p && j < sizeof(w) - 1; p++)
        w[j++] = (char)tolower((unsigned char)*p);
    w[j] = '\0';

    const char    *blob = (const char *)(g_m3 + g_m3h->off_wordblob);
    const WordRec *recs = (const WordRec *)(g_m3 + g_m3h->off_wordrec);
    long lo = 0, hi = (long)g_m3h->n_words - 1;
    while (lo <= hi) {
        long mid = (lo + hi) / 2;
        int c = strcmp(blob + recs[mid].name_off, w);
        if (c < 0)      lo = mid + 1;
        else if (c > 0) hi = mid - 1;
        else {
            out[0] = recs[mid].eng_idx;
            out[1] = recs[mid].wn_idx;
            out[2] = recs[mid].phon_idx;
            if (degree && out[0] >= 0) {
                const uint32_t *rp = (const uint32_t *)(g_m3 + g_m3h->off_rowptr);
                *degree = rp[out[0] + 1] - rp[out[0]];
            }
            return 1;
        }
    }
    return 0;
}

/* ptol.c:193-202 (constants copied with it) */
static const int GR_CONTEXT_PRIMES[19] = {73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167};
#define GR_HYPONYMS_IDX 2
static double gr_log_anchor(void)
{
    static double anchor = -1.0;
    if (anchor < 0.0) {
        anchor = 0.0;
        for (int i = 0; i < 19; i++)
            if (i != GR_HYPONYMS_IDX) anchor += log((double)GR_CONTEXT_PRIMES[i]);
    }
    return anchor;
}

/* ptol.c:206-213 */
static double monad3_gamma_radial(const int16_t vec[19])
{
    double log_code = 0.0;
    for (int i = 0; i < 19; i++)
        if (vec[i]) log_code += (double)vec[i] * log((double)GR_CONTEXT_PRIMES[i]);
    if (log_code <= 0.0) return NAN;
    return tanh(0.5 * log(log_code / gr_log_anchor()));
}

int main(void) {
    int fd = open("../../PtolC/monad3_c.bin", O_RDONLY); struct stat st; fstat(fd, &st);
    g_m3 = mmap(NULL, st.st_size, PROT_READ, MAP_PRIVATE, fd, 0); g_m3h = (const Monad3cHeader *)g_m3;
    const char *words[] = {"tree", "what", "piano", "xyzzyq"};
    int hits = 0;
    for (int i = 0; i < 4; i++) {
        int32_t ix[3]; uint32_t deg; int hit = monad3_lookup(words[i], ix, &deg);
        printf("%-8s hit=%d  eng=%d wn=%d phon=%d  degree=%u", words[i], hit, ix[0], ix[1], ix[2], deg);
        if (hit && ix[1] >= 0) {
            const int16_t *vec = (const int16_t *)(g_m3 + g_m3h->off_wn + (size_t)ix[1] * 82u + 40);
            printf("   gamma_radial=%+.6f", monad3_gamma_radial(vec));
        }
        printf("\n"); if (hit) hits++;
    }
    printf("%s: three of the four words are in the store and the unknown word is not (hits = %d)\n", hits == 3 ? "PASS" : "FAIL", hits);
    return 0;
}


tree     hit=1  eng=563 wn=134483 phon=112979  degree=292   gamma_radial=-0.904887


what     hit=1  eng=228 wn=-1 phon=119807  degree=2697
piano    hit=1  eng=21590 wn=99564 phon=84465  degree=20   gamma_radial=+nan
xyzzyq   hit=0  eng=-1 wn=-1 phon=-1  degree=0
PASS: three of the four words are in the store and the unknown word is not (hits = 3)


## Every word gets a slot and an energy between the two constants

In [3]:
//%cflags:-I../../PtolC
//%ldflags:-lm
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <math.h>
#include "ptolemy.h"

/* monad.c:170-220 */
void monad_word_coords(const char *surface, int N, int *idx, double *E)
{
    uint64_t v = 0;
    for (const char *p = surface; *p; p++) {
        unsigned char c = (unsigned char)*p;
        int ci = (c >= 32 && c < 127) ? (int)(c - 32 + 1) : 0;
        v = v * 95ULL + (uint64_t)ci;
    }
    if (v > 0) v--;

    /* Fibonacci (Knuth multiplicative) hashing — exact in integer arithmetic.
     *
     * WAS:  double seed = fmod((double)v * MONAD_PHI, 1.0);
     *
     * That converted v to double before taking the fractional part. A double
     * carries a 53-bit mantissa: 95^8 = 6.63e15 fits under 2^53 = 9.01e15,
     * 95^9 = 6.30e17 does not. Past 2^53 the low-order bits — the only bits
     * fmod(.,1.0) depends on — were gone and seed collapsed to exactly 0.0.
     * Every word of 8+ chars therefore landed on zero 0 with E = D_STAR
     * (the MINIMUM E), and then lost the `E > vocab[idx].E` contest below.
     * Measured on WordNet: 62,961 of 101,916 unique tokens piled onto idx 0;
     * survival was 0.00% for every length >= 8.
     *
     * PHI_64 = round(2^64 / phi) = 0x9E3779B97F4A7C15. Since phi = 1 + 1/phi
     * and v is an integer, frac(v*phi) == frac(v/phi) — this computes the
     * same quantity the line above was reaching for, but exactly. The uint64
     * multiply wraps, which is defined and is what a hash wants; the double
     * conversion now happens only at the end, on a value already in
     * [0, 2^64), and just ~15 of its bits are needed to pick 1 of N zeros.
     *
     * Verified before this edit (VAPMIP/monad_addressing.py, 101,916 tokens):
     *   old: chi2=3833102 z=+272400  KS D=0.6178 NOT UNIFORM  max pileup 62961
     *   new: chi2=100.0   z=+0.1     KS D=0.0025 UNIFORM      max pileup 14
     *   occupied zeros 14173 -> 24590 (expected 24576)
     *
     * NOTE: this is a HASH, not a bijection, and always was. 101,916 tokens
     * into N=25,000 zeros collide ~4:1 by pigeonhole. This removes the LENGTH
     * BIAS; it does not raise the ceiling.
     *
     * BREAKING: word addresses change for ~83% of tokens (everything from
     * length 6 up). Existing .bin checkpoints are NOT compatible and must be
     * rebuilt from corpus. */
    uint64_t h    = v * 0x9E3779B97F4A7C15ULL;
    double   seed = (double)h / 18446744073709551616.0;   /* h / 2^64 */

    *idx = (int)(seed * N);
    if (*idx >= N) *idx = N - 1;
    if (*idx < 0)  *idx = 0;

    *E = MONAD_D_STAR + seed * (MONAD_OMEGA_ZS - MONAD_D_STAR);
}

int main(void) {
    const char *words[] = {"tree", "what", "piano", "neutron", "good", "morning", "reactor", "discovered"};
    double lo = 1e9, hi = -1e9; int N = 25000;
    for (int i = 0; i < 8; i++) {
        int idx; double E; monad_word_coords(words[i], N, &idx, &E);
        printf("%-12s slot %5d  E = %.6f\n", words[i], idx, E);
        if (E < lo) lo = E; if (E > hi) hi = E;
    }
    printf("MONAD_D_STAR = %.5f, MONAD_OMEGA_ZS = %.5f; observed E in [%.6f, %.6f]\n", MONAD_D_STAR, MONAD_OMEGA_ZS, lo, hi);
    printf("%s: every energy lies between MONAD_D_STAR and MONAD_OMEGA_ZS\n", (lo >= MONAD_D_STAR - 1e-12 && hi <= MONAD_OMEGA_ZS + 1e-12) ? "PASS" : "FAIL");
    return 0;
}


tree         slot  3109  E = 0.285940
what         slot 16194  E = 0.454032
piano        slot  2639  E = 0.279904
neutron      slot  7936  E = 0.347950
good         slot 19600  E = 0.497781
morning      slot 16537  E = 0.458432
reactor      slot 18415  E = 0.482558
discovered   slot 23100  E = 0.542734
MONAD_D_STAR = 0.24600, MONAD_OMEGA_ZS = 0.56714; observed E in [0.279904, 0.542734]
PASS: every energy lies between MONAD_D_STAR and MONAD_OMEGA_ZS


## The slots printed in two screenshots of May 2026 (paper §7.3, screenshots 2 and 3)
The ten (word, slot, energy) triples below are transcribed from the screenshots `2-Hocus-Hyperwebster.png` and `3-TheExtractor.png`, taken on the earlier store (`monad_wordnet.bin`, vocab 24,485); the function is today's `monad_word_coords`.

In [4]:
//%cflags:-I../../PtolC
//%ldflags:-lm
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <math.h>
#include "ptolemy.h"

void monad_word_coords(const char *surface, int N, int *idx, double *E)
{
    uint64_t v = 0;
    for (const char *p = surface; *p; p++) {
        unsigned char c = (unsigned char)*p;
        int ci = (c >= 32 && c < 127) ? (int)(c - 32 + 1) : 0;
        v = v * 95ULL + (uint64_t)ci;
    }
    if (v > 0) v--;

    /* Fibonacci (Knuth multiplicative) hashing — exact in integer arithmetic.
     *
     * WAS:  double seed = fmod((double)v * MONAD_PHI, 1.0);
     *
     * That converted v to double before taking the fractional part. A double
     * carries a 53-bit mantissa: 95^8 = 6.63e15 fits under 2^53 = 9.01e15,
     * 95^9 = 6.30e17 does not. Past 2^53 the low-order bits — the only bits
     * fmod(.,1.0) depends on — were gone and seed collapsed to exactly 0.0.
     * Every word of 8+ chars therefore landed on zero 0 with E = D_STAR
     * (the MINIMUM E), and then lost the `E > vocab[idx].E` contest below.
     * Measured on WordNet: 62,961 of 101,916 unique tokens piled onto idx 0;
     * survival was 0.00% for every length >= 8.
     *
     * PHI_64 = round(2^64 / phi) = 0x9E3779B97F4A7C15. Since phi = 1 + 1/phi
     * and v is an integer, frac(v*phi) == frac(v/phi) — this computes the
     * same quantity the line above was reaching for, but exactly. The uint64
     * multiply wraps, which is defined and is what a hash wants; the double
     * conversion now happens only at the end, on a value already in
     * [0, 2^64), and just ~15 of its bits are needed to pick 1 of N zeros.
     *
     * Verified before this edit (VAPMIP/monad_addressing.py, 101,916 tokens):
     *   old: chi2=3833102 z=+272400  KS D=0.6178 NOT UNIFORM  max pileup 62961
     *   new: chi2=100.0   z=+0.1     KS D=0.0025 UNIFORM      max pileup 14
     *   occupied zeros 14173 -> 24590 (expected 24576)
     *
     * NOTE: this is a HASH, not a bijection, and always was. 101,916 tokens
     * into N=25,000 zeros collide ~4:1 by pigeonhole. This removes the LENGTH
     * BIAS; it does not raise the ceiling.
     *
     * BREAKING: word addresses change for ~83% of tokens (everything from
     * length 6 up). Existing .bin checkpoints are NOT compatible and must be
     * rebuilt from corpus. */
    uint64_t h    = v * 0x9E3779B97F4A7C15ULL;
    double   seed = (double)h / 18446744073709551616.0;   /* h / 2^64 */

    *idx = (int)(seed * N);
    if (*idx >= N) *idx = N - 1;
    if (*idx < 0)  *idx = 0;

    *E = MONAD_D_STAR + seed * (MONAD_OMEGA_ZS - MONAD_D_STAR);
}

int main(void) {
    const char *w[] = {"can","you","hear","what","i","am","typing","is","your","name"};
    int z[] = {18466,17464,19372,16194,2912,16543,21041,1894,18893,15471};
    double E[] = {0.4832,0.4703,0.4949,0.4540,0.2834,0.4585,0.5163,0.2703,0.4887,0.4447};
    int old_ok = 1, new_same = 0;
    for (int i = 0; i < 10; i++) {
        int idx; double e; monad_word_coords(w[i], 25000, &idx, &e);
        /* the formula the comment in monad_word_coords says it replaced: fmod((double)v*MONAD_PHI, 1.0) */
        uint64_t v = 0; for (const char *p = w[i]; *p; p++) v = v * 95ULL + (uint64_t)((unsigned char)*p - 32 + 1);
        if (v > 0) v--;
        int old_idx = (int)(fmod((double)v * MONAD_PHI, 1.0) * 25000);
        int match_new = (idx == z[i]) && (fabs(e - E[i]) < 5e-5), match_old = (old_idx == z[i]);
        if (!match_old) old_ok = 0;
        if (match_new) new_same++;
        printf("shot %-7s screen slot %5d | today %5d | old formula %5d | E %.4f (screen %.4f) %s\n", w[i], z[i], idx, old_idx, e, E[i], match_new ? "match" : "DIFFERS (today)");
    }
    printf("%s: the replaced formula (fmod of v*MONAD_PHI) reproduces all ten screenshot slots; today's exact-integer monad_word_coords reproduces %d of 10 (the others differ by one slot, six-character word, as monad.c's own comment says)\n", (old_ok && new_same >= 9) ? "PASS" : "FAIL", new_same);
    return 0;
}


shot can     screen slot 18466 | today 18466 | old formula 18466 | E 0.4832 (screen 0.4832) match
shot you     screen slot 17464 | today 17464 | old formula 17464 | E 0.4703 (screen 0.4703) match
shot hear    screen slot 19372 | today 19372 | old formula 19372 | E 0.4949 (screen 0.4949) match
shot what    screen slot 16194 | today 16194 | old formula 16194 | E 0.4540 (screen 0.4540) match
shot i       screen slot  2912 | today  2912 | old formula  2912 | E 0.2834 (screen 0.2834) match
shot am      screen slot 16543 | today 16543 | old formula 16543 | E 0.4585 (screen 0.4585) match
shot typing  screen slot 21041 | today 21042 | old formula 21041 | E 0.5163 (screen 0.5163) DIFFERS (today)
shot is      screen slot  1894 | today  1894 | old formula  1894 | E 0.2703 (screen 0.2703) match
shot your    screen slot 18893 | today 18893 | old formula 18893 | E 0.4887 (screen 0.4887) match
shot name    screen slot 15471 | today 15471 | old formula 15471 | E 0.4447 (screen 0.4447) match
PASS: the 